# Mini-TP 3 — Sirve tu modelo por gRPC (starter)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual, esta semana**

Expón **tu propio modelo** (el mismo que serviste por REST en la Sesión 1) como un servicio **gRPC**. Este notebook es un *starter*: completa las celdas marcadas con `# TODO`.

**Qué debes entregar**
1. Un `scoring.proto` con un servicio para tu modelo (mensajes de entrada y salida **tipados**).
2. Los *stubs* generados y un **servidor** que cargue tu modelo **una sola vez**.
3. Un **cliente** que llame al servicio (**unary**).
4. Un método de **server-streaming** que puntúe un lote.
5. Una **comparación de latencia** contra tu endpoint REST de la Sesión 1, con una breve reflexión.

**Se evalúa:** que corra de punta a punta; que el `.proto` tipe entrada y salida; que funcionen unary y streaming; y la reflexión gRPC vs REST.

> Apóyate en `grpc_tutorial.ipynb`, que hace todo esto paso a paso. Entorno con uv:
> `uv add grpcio grpcio-tools scikit-learn joblib requests` (y lo que use tu modelo).

In [1]:
# === Configuracin de conexin a MLflow + MinIO ===
# El tracking server responde con una URI s3://... y boto3 descarga los artefactos
# directo de MinIO, as que hacen falta estas credenciales (contenedor mlops-minio).
import os

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin123"

import mlflow
mlflow.set_tracking_uri("http://localhost:5000")

In [2]:
# Dependencias (equivalente uv: uv add grpcio grpcio-tools joblib ...)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "grpcio", "grpcio-tools", "scikit-learn", "joblib", "requests"])
print("Listo.")

Listo.


## 1. Tu modelo

Carga aquí **tu** modelo entrenado (por ejemplo `model.pkl`). Si todavía no lo tienes a mano, entrena uno rápido para probar la mecánica.

In [2]:
# Carga tu modelo desde el Model Registry. Debe exponer .predict(X)
# (la conexin y credenciales ya se configuraron en la celda de arriba)
import pickle
import numpy as np

# v2 = modelo re-guardado en formato JSON (la v1 estaba en formato binario .xgb,
#      deprecado y no cargable con xgboost >= 3.1)
model = mlflow.xgboost.load_model("models:/xgb_best/2")   # o "models:/xgb_best/latest"
print("Modelo:", type(model).__name__)

# Serializar a .pkl (para el mini-TP: el servidor gRPC lo carga una sola vez)
with open("xgb_best.pkl", "wb") as f:
    pickle.dump(model, f)

modelo = model
N_FEATURES = model.n_features_in_
print("Modelo cargado. n_features =", N_FEATURES)
print("Prediccin de prueba:", modelo.predict(np.zeros((1, N_FEATURES))))

Modelo: XGBRegressor
Modelo cargado. n_features = 23
Prediccin de prueba: [12.191017]


## 2. El contrato `.proto`

Define el servicio para tu modelo. Debe tener **al menos** un método unary y uno de streaming.

In [3]:
%%writefile scoring.proto
syntax = "proto3";
package scoring;

// Una fila de entrada: vector de features de TU modelo (XGBRegressor, N_FEATURES=23)
// + el nombre/id del modelo que se quiere invocar (permite servir varios modelos).
message Features {
  repeated double values = 1;
  string model = 2;
}

// Un lote de filas, para el método de server-streaming.
message Batch {
  repeated Features items = 1;
}

message Prediction {
  double score = 1;
  string model_version = 2;
}

service Scoring {
  rpc Predict(Features) returns (Prediction);            // unary: 1 -> 1
  rpc PredictStream(Batch) returns (stream Prediction);   // server streaming: N filas -> N predicciones
}

Overwriting scoring.proto


## 3. Genera los *stubs*

In [4]:
import sys, subprocess, os
subprocess.run([sys.executable, "-m", "grpc_tools.protoc",
                "-I.", "--python_out=.", "--grpc_python_out=.", "scoring.proto"], check=True)
print("Generados:", [f for f in os.listdir('.') if f.startswith('scoring_pb2')])

Generados: ['scoring_pb2.py', 'scoring_pb2_grpc.py']


## 4. El servidor

Implementa los métodos. Recuerda: el modelo se carga **una vez** (ya lo cargaste arriba).

In [5]:
import grpc, threading
from concurrent import futures
import scoring_pb2, scoring_pb2_grpc

class ScoringServicer(scoring_pb2_grpc.ScoringServicer):
    def _predict_one(self, features, context):
        """Valida y puntúa un único Features. Devuelve None si hubo error (ya seteado en context)."""
        if len(features.values) != N_FEATURES:
            context.set_code(grpc.StatusCode.INVALID_ARGUMENT)
            context.set_details(
                f"Se esperaban {N_FEATURES} valores, llegaron {len(features.values)}."
            )
            return None
        x = np.array(features.values, dtype=float).reshape(1, -1)
        y = modelo.predict(x)[0]
        return float(y)

    def Predict(self, request, context):
        y = self._predict_one(request, context)
        if y is None:
            return scoring_pb2.Prediction()
        return scoring_pb2.Prediction(score=y, model_version="v1")

    def PredictStream(self, request, context):
        # request es un Batch: itera cada fila y va emitiendo su predicción (server-streaming)
        for features in request.items:
            y = self._predict_one(features, context)
            if y is None:
                return  # corta el stream; el error ya quedó seteado en el context
            yield scoring_pb2.Prediction(score=y, model_version="v1")

servidor = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
scoring_pb2_grpc.add_ScoringServicer_to_server(ScoringServicer(), servidor)
servidor.add_insecure_port("[::]:50051")
servidor.start()
print("Servidor gRPC en localhost:50051")

Servidor gRPC en localhost:50051


## 5. El cliente (unary)

In [6]:
canal = grpc.insecure_channel("localhost:50051")
stub = scoring_pb2_grpc.ScoringStub(canal)

# Entrada vlida para tu modelo: N_FEATURES valores (aqu, un ejemplo dummy)
import numpy as np
valores = np.random.RandomState(0).rand(N_FEATURES).tolist()
entrada = scoring_pb2.Features(values=valores, model="xgb_best")
r = stub.Predict(entrada)
print("score:", r.score, "| version:", r.model_version)

score: 12.191017150878906 | version: v1


## 6. Streaming

Una vez que agregaste `PredictStream` al `.proto`, lo regeneraste y lo implementaste en el servidor, pruébalo aquí.

In [8]:
# TODO: descomenta cuando tengas PredictStream implementado

lote = scoring_pb2.Batch(items=[
    scoring_pb2.Features(
        values=np.random.RandomState(i).rand(N_FEATURES).tolist(),
        model="xgb_best",
    )
    for i in range(5)
])

for p in stub.PredictStream(lote):
    print(round(p.score, 3))
#print("Pendiente: implementar y probar el streaming.")

12.191
12.191
12.191
12.191
12.191


## 7. Comparación de latencia gRPC vs REST

Levanta (o apunta a) tu REST de la Sesión 1 y mide el tiempo medio por llamada. Anota abajo tu reflexión.

In [9]:
import time
N = 10
feat = scoring_pb2.Features(values=valores)   # mismos N_FEATURES valores de la celda del cliente
t = time.perf_counter()
for _ in range(N):
    r = stub.Predict(feat)
    #print('Resultado de la predicción gRPC:', format(r.score, '.3f'))
grpc_ms = (time.perf_counter() - t) / N * 1000
print(f"gRPC: {grpc_ms:.3f} ms/llamada")

# Mide tu endpoint REST: mismo modelo, mismos `valores` (23 features) que gRPC recibió.
from fastapi import FastAPI, APIRouter
from pydantic import BaseModel
import threading, time, uvicorn

app = FastAPI(title="ML API - REST")

class FeaturesIn(BaseModel):
    values: list[float]

class ModeloV1():
    def predecir(self, values: list[float]):
        x = np.array(values, dtype=float).reshape(1, -1)
        return float(modelo.predict(x)[0])

models_list = {"v1": ModeloV1()}

router_v1 = APIRouter(prefix="/v1", tags=["Modelo V1"])
@router_v1.post("/rest/models/{mname}/predict")
def rest_predict(mname: str, payload: FeaturesIn):
    return models_list["v1"].predecir(payload.values)

app.include_router(router_v1)
def _run():
    uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8020, log_level="warning")).run()
threading.Thread(target=_run, daemon=True).start()
print("Servidor REST listo y atendiendo en localhost:8020")
# Consultamos el servidor REST en la proxima celda

gRPC: 1.548 ms/llamada
Servidor REST listo y atendiendo en localhost:8020


In [10]:
import requests, time
t = time.perf_counter()
for _ in range(N):
    r = requests.post("http://localhost:8020/v1/rest/models/xgb_best/predict",
                       json={"values": valores})
    #print('Resultado de la predicción REST:', format(r.json(), '.3f'))
rest_ms = (time.perf_counter() - t) / N * 1000

print(f"REST: {rest_ms:.3f} ms/llamada  ->  gRPC ~{rest_ms/grpc_ms:.1f}x")


REST: 2032.347 ms/llamada  ->  gRPC ~1313.2x


### Reflexión (completa)

_TODO: 3–5 líneas. ¿Qué diferencia de latencia observaste? ¿Cuándo usarías gRPC y cuándo REST en tu plataforma? ¿Qué costo tiene gRPC (contrato, tooling, no lo habla el navegador directo)?_

Con N=10 llamadas, gRPC resultó ~1472x más rápido que REST (1.385 ms/llamada vs 2038.363 ms/llamada). Parte de esa ventaja viene de HTTP/2 (multiplexado, sin overhead de reconexión), pero también pesa la serialización binaria de Protocol Buffers frente a JSON, y el hecho de reusar un único canal/stub gRPC persistente durante todo el loop, mientras que cada `requests.post` de REST paga el costo de una request HTTP/1.1 independiente. Otro factor que a mi criterio es el mas importante y explica la diferencia tan grande es que el server REST y el cliente corren en el mismo proceso/thread del kernel del notebook, provocando una contención entre el loop de uvicorn en el thread daemon y el requests.post síncrono, que en máquinas diferentes (o incluso threads diferentes corriendo el servidor por fuera del notebook) no sucederia.

Cuando es muy importante el tiempo de respuesta, para aplicaciones de misión crítica o para grandes volúmenes de peticiones, la latencia se vuelve un factor clave y gRPC es mucho más conveniente. En cambio, para APIs públicas o integración con terceros, donde no tenemos control de la otra parte, REST es más apropiado porque es un estándar, de más fácil integración y por tanto mejor interoperabilidad, en detrimento del desempeño y performance.

El costo de gRPC es mayor en cuanto al contrato que debe construirse, el cual será más rígido, aumentando la confiabilidad del sistema en general, aunque con mayor tiempo de implementación y de mantenimiento futuro. Lo mismo sucede con el tooling necesario (stubs y proto), que es necesario generar y mantener sincronizado con el código. Al estar implementado sobre HTTP/2, gRPC aprovecha bien sus ventajas para ofrecer alto rendimiento en las llamadas, aunque a cambio las pruebas no pueden hacerse directamente desde un navegador como sucede con REST, lo cual complica el debugging manual y la exploración ad-hoc del servicio.


In [ ]:
# limpieza
canal.close(); servidor.stop(0)
print("Servidor detenido.")